# January 2026 wind forecast error analysis

Reproduces the README findings from the cached data using the same `core` functions the API uses. Run from the repo root with the backend venv: `backend/.venv/bin/pip install jupyter matplotlib && backend/.venv/bin/jupyter notebook notebooks/analysis.ipynb`.

In [ ]:
import sys
from datetime import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "backend"))
import pandas as pd
from app.config import get_settings
from app.core.data_store import get_store
from app.core.metrics import by_horizon, by_hour, summarize
from app.core.selection import build_series

s = get_settings()
store = get_store()
A, F, START, END = store.actuals, store.forecasts, s.target_start, s.target_end

## Headline metrics: 4 h horizon vs D-1 09:00 decision time

In [ ]:
h4 = build_series(A, F, START, END, "horizon", 4)
dec = build_series(A, F, START, END, "decision", cutoff=time(9))
pd.DataFrame({"4h": summarize(h4).to_dict(), "D-1 09:00": summarize(dec).to_dict()})

## Error vs horizon
Coverage drops beyond ~43 h: runs publish at :30 every 2-4 h, so with a hard 48 h cap only some targets have a run published in [T-48h, T-H].

In [ ]:
hz = pd.DataFrame(by_horizon(A, F, START, END)).set_index("horizon")
hz[["mae", "bias", "p10"]].plot(figsize=(9, 4), grid=True, title="Error vs horizon (MW)");
hz

## Where does the bias come from?
The forecast is above outturn most of the time and the gap grows with forecast level and overnight, consistent with curtailment / constraint actions (or a fleet-definition difference) rather than pure weather error.

In [ ]:
dec["band"] = pd.cut(dec.forecast, [0, 5000, 10000, 15000, 25000])
print("share over-forecast:", round((dec.error > 0).mean(), 3))
dec.groupby("band", observed=True)["error"].agg(["mean", "count"]).round(0)

In [ ]:
pd.DataFrame(by_hour(dec)).set_index("hour")[["mae", "bias"]].plot.bar(figsize=(10, 4), title="D-1 decision mode by UTC hour");

## Safety margin if the bias were corrected
Illustrative only: a de-biased forecast (subtract the in-sample mean bias) gives the *random* part of the margin. In-sample, so optimistic.

In [ ]:
short = (dec.actual - dec.forecast).dropna()
print("P10 raw:       ", round(short.quantile(0.10)))
print("P10 de-biased: ", round((short - short.mean()).quantile(0.10)))